In [4]:
import pandas as pd
import re
import nltk
from pymorphy3 import MorphAnalyzer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

morph = MorphAnalyzer()
stopwords_ru = stopwords.words("russian")

def preprocess_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'[^а-яa-z0-9\s]', ' ', text) 
    words = text.split()
    lemmatized = [morph.parse(word)[0].normal_form for word in words]
    clean_words = [w for w in lemmatized if w not in stopwords_ru and len(w) > 2]
    return " ".join(clean_words)

try:
    df_log = pd.read_csv('../data/history_log.csv')
    print(f"Всего записей в логе: {len(df_log)}")
except FileNotFoundError:
    print("Файл логов не найден!")

#только те сообщения, где pattern_found == False
df_unknown = df_log[df_log['pattern_found'] == False].copy()

print(f"Сообщений для анализа (не распознанных): {len(df_unknown)}")

print("Выполняется очистка текста...")
df_unknown['processed_text'] = df_unknown['original_text'].apply(preprocess_text)

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df_unknown['processed_text'])

NUM_CLUSTERS = 7
kmeans = KMeans(n_clusters=NUM_CLUSTERS, n_init='auto', random_state=42)
df_unknown['cluster_id'] = kmeans.fit_predict(X)

print(f"\n{'='*40}")
print("РЕЗУЛЬТАТ ПОИСКА НОВЫХ ПАТТЕРНОВ")
print(f"{'='*40}")

cluster_counts = df_unknown['cluster_id'].value_counts()

for cluster_id in cluster_counts.index:
    cluster_size = cluster_counts[cluster_id]
    
    examples = df_unknown[df_unknown['cluster_id'] == cluster_id]['original_text'].head(5).tolist()
    
    print(f"\n📂 КЛАСТЕР #{cluster_id} (Размер: {cluster_size} сообщений)")
    print("-" * 30)
    for text in examples:
        print(f"• {text}")

Всего записей в логе: 3117
Сообщений для анализа (не распознанных): 2359
Выполняется очистка текста...

РЕЗУЛЬТАТ ПОИСКА НОВЫХ ПАТТЕРНОВ

📂 КЛАСТЕР #0 (Размер: 806 сообщений)
------------------------------
• Сетевая недоступность хоста 192.168.1.1
• Сетевая недоступность хоста 192.168.1.1
• Сетевая недоступность хоста 192.168.1.1
• Сетевая недоступность хоста 192.168.1.1
• Сетевая недоступность хоста 192.168.1.1

📂 КЛАСТЕР #1 (Размер: 781 сообщений)
------------------------------
• Тестовое
• Привет, это тестовое сообщение
• Привет, это тестовое сообщение
• Привет, это тестовое сообщение
• Привет, это тестовое сообщение

📂 КЛАСТЕР #2 (Размер: 746 сообщений)
------------------------------
• В системе упали поды кубернетеса в неймспейсе прод
• В системе упали поды кубернетеса в неймспейсе прод
• В системе упали поды кубернетеса в неймспейсе прод
• В системе упали поды кубернетеса в неймспейсе прод
• В системе упали поды кубернетеса в неймспейсе прод

📂 КЛАСТЕР #4 (Размер: 8 сообщений)
--